In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/mSHIFT_SHeS/code_ocean/

/content/drive/MyDrive/mSHIFT_SHeS/code_ocean


The notebook contains the steps to extract the variables  for  running the health simulation mSHIFT modules, from the participant level SHeS 2021 data.  

In [ ]:
# import necessary libraries
import numpy as np
import pandas as pd
from pathlib import Path
import sys

data_path = Path("data")

In [ ]:
sys.path.append(str(Path().resolve() / 'code/mSHIFT'))
sys.path.append(str(Path().resolve() / 'code/notebooks/notebook_code'))

In [ ]:
from health_data_processing import *

In [ ]:
core_data_path = 'data/shes_data_raw/shes21i_eul.csv'
diet_data_path =  'data/shes_data_raw/SHeS_2021_food_level_condensed.csv'

diet_data = load_data(diet_data_path)
core_data = load_data(core_data_path)

In [ ]:
df_baseline = pd.read_parquet(data_path / 'df_baseline.parquet')

## Add the physical activity variables

In [ ]:
PA_variables = ['MVPA10wk', 'hrshwk10', 'hrsman10', 'hrwalk10R', 'actwktime']

for var in PA_variables:
  df_baseline[var] = df_baseline.apply(lambda row: add_variable(row, variable=var, core_data=core_data), axis=1)

In [ ]:
df_baseline = df_baseline.replace({'actwktime' : {'Not applicable': 0}})

In [ ]:
# Assume that respondents who answered "Don't know" to minutes of physical activity have zero minutes
df_baseline['hrwalk10R'] = df_baseline['hrwalk10R'].replace("Don't know", 0.0)
df_baseline['actwktime'] = df_baseline['actwktime'].replace("Don't know", 0.0)

In [ ]:
for var in ['hrshwk10', 'hrsman10', 'hrwalk10R', 'actwktime']:
  df_baseline.replace({var : {'Not applicable': 0}})
  df_baseline[var] = df_baseline[var].astype(float)
  df_baseline[var] = df_baseline[var]*60

In [ ]:
df_baseline['MVPA10wk'] = df_baseline['MVPA10wk'].astype(float)

In [ ]:
# Definition of moderate and vigorous physical activity varaibles in terms of the SHeS variables
df_baseline['Vigorous PA'] = df_baseline[['MVPA10wk', 'hrshwk10', 'hrsman10']].sum(axis=1)
df_baseline['Moderate PA'] = df_baseline[['hrwalk10R', 'actwktime']].sum(axis=1)

In [ ]:
df_baseline["Avg Mins MVPA per week"] = df_baseline['Vigorous PA'] + df_baseline['Moderate PA']

In [ ]:
# Convert minutes of physical activity per week to minutes of of physical activity per day as required by the obesity model
df_baseline['Vigorous PA'] /= 7
df_baseline['Moderate PA'] /= 7

In [ ]:
# Assume that all minutes of the day not encoded as either moderate or vigorous physical activity were sedentary minutes
df_baseline['Sedentary'] = 1440 - (df_baseline['Vigorous PA'] + df_baseline['Moderate PA'])

# Assume 7 hours of sleep per individual
df_baseline['Sedentary non-sleep'] = df_baseline['Sedentary']-420
df_baseline['Sedentary sleep']=420

In [ ]:
df_baseline[df_baseline['Sedentary non-sleep'] < 0]

,Energykcal,EnergykJ,Proteing,Fatg,Carbohydrateg,Sodiummg,Potassiummg,Calciummg,Magnesiummg,Phosphorusmg,...,Sedentary non-sleep,Vigorous PA,Moderate PA,Sedentary,MVPA10wk,hrshwk10,hrsman10,hrwalk10R,actwktime,Sedentary sleep
1400004901,3619.260,15199.360,103.000,71.93,357.520,3192.680,5312.290,969.11,854.810,2313.570,...,-12.857143,390.000000,642.857143,407.142857,420.0,630.0,1680.0,2100.0,2400.0,420
1400845501,681.750,2868.995,25.000,23.93,98.175,927.155,1202.435,423.59,98.235,508.650,...,-681.428571,501.428571,1200.000000,-261.428571,2370.0,420.0,720.0,8400.0,0.0,420
1402380201,794.555,3334.010,46.260,33.40,74.885,923.165,1895.890,397.77,146.515,683.745,...,-9.285714,617.857143,411.428571,410.714286,4175.0,150.0,0.0,480.0,2400.0,420
1404296502,2550.735,10681.145,109.225,115.35,233.125,3185.250,2981.955,1714.64,496.190,2164.500,...,-107.142857,724.285714,402.857143,312.857143,5040.0,30.0,0.0,420.0,2400.0,420


## Rescale the sample weights such that their sum is equal to adult population size

In [ ]:
## Sum of all population sizes in age groups <16 years from mid-year-pop-est-21-data_SHeS_population.xlsx
young_ppl = 46782 +49017 +51478+53317+54843+57070+57945 + 58262+59490+60960 + 62868 +59950 + 61557 +61334+58857 +57792

In [ ]:
# 5479900 estimated total population size from the 2021 census data
total_pop = 5479900 - young_ppl

In [ ]:
# Rescale the sample weights such that their sum is equal to the total Scottish adult population size
sample_weight_scale = total_pop / 3447

In [ ]:
df_baseline['Sample Weight scaled'] = df_baseline['Samlple Weight scaled']*sample_weight_scale

In [ ]:
# Check that the sample weights sum to ~4.5 million
df_baseline['Sample Weight scaled'].sum()

np.float64(3446.9999999789998)

In [ ]:
df_baseline.to_parquet(data_path / 'df_baseline.parquet')

## Calculate red and processed meat intake

In [ ]:
foods_RM = ['Beefg', 'Lambg', 'Burgersg', 'Porkg', 'OtherRedMeatg'] # check if burgers are processed or unprocessed red meat
foods_RPM = ['ProcessedRedMeatg', 'Sausagesg', 'Offalg'] # some offal items are not processed red meat: need to manually exclude them

In [ ]:
df_baseline['Red meat intake'] = df_baseline[foods_RM].sum(axis=1)
df_baseline['Processed meat intake'] = df_baseline[foods_RPM].sum(axis=1) - df_baseline['white PM intake']

## Extract the variables for the health simulation

In [ ]:
df = df_baseline[['Red meat intake', 'Processed meat intake', 'SIMD1',
'SIMD2',
'SIMD3',
'SIMD4',
'SIMD5',
'white_scot',
'white_OB',
'asian',
'white_oth',
'oth_min_eth' ,
'Ethnicity',
'age', 'Sex',
'BMI', 'Sample Weight',
'Vigorous PA', 'Moderate PA','Sedentary non-sleep', 'Sedentary sleep', 'Avg Mins MVPA per week']]

In [ ]:
df['height'] = df.apply(lambda row: add_variable(row, variable = 'SlfHtDV_adj',  core_data=core_data), axis=1)
df['weight'] = df.apply(lambda row: add_variable(row, variable = 'SlfWtDV_adj',  core_data=core_data), axis=1)
df['initial weight'] = df['weight'].copy()

In [ ]:
df['Current Smoker'] =  df.apply(lambda row: add_variable(row, variable = 'Cignow',  core_data=core_data), axis=1)

In [ ]:
df['Diabetes'] = df.apply(lambda row: add_variable(row, variable = 'Type2',  core_data=core_data), axis=1)
df['Parental Diabetes History'] = df.apply(lambda row: add_variable(row, variable = 'FamDB', core_data=core_data), axis=1)
df['CVD'] = df.apply(lambda row: add_variable(row, variable = 'cvddef1',  core_data=core_data), axis=1)
df['preg'] =  df.apply(lambda row: add_variable(row, variable = 'PregNTJ',  core_data=core_data), axis=1)
df['Current Smoker'] =  df.apply(lambda row: add_variable(row, variable = 'Cignow',  core_data=core_data), axis=1)
df['Taking BPM'] =  df.apply(lambda row: add_variable(row, variable = 'medcinbp',  core_data=core_data), axis=1)
df['High BP'] =  df.apply(lambda row: add_variable(row, variable = 'currbp', core_data=core_data), axis=1)

In [ ]:
df = df.replace({'Diabetes': {'No': 0, "Don't know": 0, "Yes": 1},
            'Parental Diabetes History':{'No': 0, "Don't know": 0, "Yes": 1, 'Refused': 0},
            'CVD': {'No': 0, "Don't know": 0, "Yes": 1},
            'preg': {'No': 0, "Don't know": 0, "Yes": 1, 'Not applicable': 0},
            'Current Smoker': {'No': 0, "Don't know": 0, "Yes": 1, 'Not applicable': 0},
            'Taking BPM' :  {'No': 0, "Don't know": 0, "Yes": 1, 'Not applicable': 0},
            "High BP": {'No': 0, "Don't know": 0, "Yes": 1, 'Not applicable': 0} })

<ipython-input-29-6f4af7e4c7bc>:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace({'Diabetes': {'No': 0, "Don't know": 0, "Yes": 1},


In [ ]:
# Initialise the mSHIFT variables for estimating new cases
df[['New diabetes cases',
       'New CVD cases', 'New CRC cases', 'New diabetes and CVD cases',
       'New diabetes and CRC cases', 'New CVD and CRC cases',
       'New diabetes and CVD and CRC cases']] = 0

In [ ]:
# Identify participants without CVD or diabetes at the start of the simlation
df['healthy'] = df.apply(lambda row: healthy(row), axis=1)

In [ ]:
# Check prevalance of the population without diabetes or CVD
df['healthy'].sum() / df['Sample Weight'].sum()

np.float64(0.8103398253803356)

In [ ]:
# Initialise bio variables to be imputed
df['HDL Cholesterol'] = np.nan
df['Total Cholesterol'] = np.nan
df['Systolic Blood Pressure'] = np.nan

In [ ]:
# Remove pregnant participants from the data for the health estimates.
pregnant_ids = df[df['preg'] == 1].index.tolist()
df = df[~df.index.isin(pregnant_ids)]

In [ ]:
# Save the dataframe
df.to_parquet(data_path / 'df_SHeS_unimputed.parquet')